# Milestone 6 — Evaluation

Lässt den Agent automatisch gegen unser LangSmith-Dataset laufen und bewertet die Ergebnisse mit mehreren Kriterien: Tool-Auswahl, Groundedness, Format-Konsistenz.


In [2]:
import inspect
from langchain.agents import create_agent
print(inspect.signature(create_agent))

(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None, transformers: 'Sequence[TransformerFactory] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]'


In [3]:
import sys
sys.path.append("../backend")

from langsmith import Client
from agent import ask_agent, ask_agent_with_trace

client = Client()
dataset_name = "health-fitness-eval-v1"

print("Verbunden mit LangSmith")

Verbunden mit LangSmith


## Target-Funktion: ruft unseren Agent auf


In [4]:
def target(inputs: dict) -> dict:
    question = inputs["question"]

    # Der Memory-Test-Eintrag soll bewusst im selben Thread wie die Kreatin-Fact-Check-Frage laufen
    if "following up" in question.lower():
        thread_id = "eval-creatine-thread"
    elif "creatine" in question.lower() and "true" in question.lower():
        thread_id = "eval-creatine-thread"
    else:
        # Jeder andere Eintrag bekommt eine EIGENE, isolierte Thread-ID
        thread_id = f"eval-{hash(question)}"

    result = ask_agent_with_trace(question, thread_id=thread_id)
    return result

## Evaluator 1: Tool-Auswahl

Prüft anhand des LangSmith-Traces, ob das erwartete Tool tatsächlich aufgerufen wurde.


In [5]:
def tool_selection_eval(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    expected_tool = reference_outputs.get("expected_tool", "")

    # Falls für diesen Eintrag kein Tool zwingend erwartet wird (z.B. Memory-Test)
    if not expected_tool or "keins" in expected_tool.lower():
        return {"key": "tool_selection", "score": None, "comment": "Kein Tool zwingend erwartet"}

    # Wir prüfen, ob der Tool-Name im LangSmith-Run als "verwendet" markiert wurde
    # -- outputs enthält bei uns nur die finale Antwort, daher nutzen wir eine einfache Heuristik:
    # das erwartete Tool sollte im Trace als tool_call auftauchen (das prüfen wir separat in der UI,
    # hier vereinfachen wir: prüfen ob die Antwort inhaltlich zum Tool passt)
    answer = outputs.get("answer", "")
    passed = len(answer) > 20  # Platzhalter -- echte Tool-Namen-Prüfung braucht Run-Metadata

    return {"key": "tool_selection", "score": passed, "comment": f"Erwartet: {expected_tool}"}

In [6]:
def get_tool_calls(run) -> list:
    child_runs = client.list_runs(parent_run_id=run.id)
    return [r.name for r in child_runs if r.run_type == "tool"]


def get_tool_outputs(run) -> str:
    child_runs = client.list_runs(parent_run_id=run.id)
    outputs = [str(r.outputs) for r in child_runs if r.run_type == "tool" and r.outputs]
    return "\n".join(outputs)

## Fortgeschrittener Method : run.child_runs

Der Trick: LangSmith-Evaluator-Funktionen können einen vierten, optionalen Parameter namens run entgegennehmen

das ist das komplette Objekt mit allen Details des Durchlaufs,

inklusive aller Tool-Aufrufe (genau das, was du in der UI unter "AI" → search_video_tool gesehen hattest).

Wir durchsuchen run.child_runs nach allen Schritten, deren Typ "tool" ist, und sammeln ihre Namen.


In [7]:
def tool_selection_evaluator(inputs: dict, outputs: dict, reference_outputs: dict, run) -> dict:
    expected_tool = reference_outputs.get("expected_tool", "")

    if not expected_tool or "keins" in expected_tool.lower():
        return {"key": "tool_selection", "score": None, "comment": "Kein Tool zwingend erwartet"}

    # Alle tatsächlich aufgerufenen Tool-Namen aus dem Trace sammeln
    used_tools = []

    used_tools = outputs.get("tools_used", [])

    passed = expected_tool in used_tools

    return {
        "key": "tool_selection",
        "score": passed,
        "comment": f"Erwartet: {expected_tool} | Tatsächlich genutzt: {used_tools}",
    }

In [8]:
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Evaluator 2: Groundedness (LLM-as-a-judge)

Lässt ein zweites LLM prüfen: Bleibt die Antwort bei den Fakten aus dem Video, oder wurden Dinge dazu erfunden, die nicht in den Tool-Ergebnissen standen?


In [9]:
def groundedness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict, run) -> dict:
    answer = outputs.get("answer", "")
    criteria = reference_outputs.get("criteria", "")

    tool_outputs = []

    context = outputs.get("context", "")

    # NEU: Problem A explizit vorab abfangen, bevor der Judge überhaupt gefragt wird
    if not context.strip():
        return {
            "key": "groundedness",
            "score": False,
            "comment": "NOT_GROUNDED: Kein Tool wurde aufgerufen -- Antwort basiert auf keinerlei Video-Kontext.",
        }

    # Problem B: Tool wurde genutzt, jetzt prüfen ob die Antwort DARÜBER HINAUSGEHT
    prompt = f"""Du bewertest, ob eine KI-Antwort ausschließlich auf dem gegebenen Kontext basiert (grounded) 
oder zusätzliche Informationen enthält, die NICHT im Kontext stehen (halluziniert).

KONTEXT (was die Tools tatsächlich zurückgegeben haben):
{context}

ANTWORT (was die KI daraus gemacht hat):
{answer}

ZUSÄTZLICHES KRITERIUM: {criteria}

Vergleiche Satz für Satz: Steht jede genannte Tatsache in der Antwort auch im Kontext?
Antworte NUR mit "GROUNDED" oder "NOT_GROUNDED", gefolgt von einem Doppelpunkt und einer kurzen Begründung, 
welche konkrete Information (falls vorhanden) NICHT im Kontext stand."""

    judge_response = judge_llm.invoke(prompt).content
    passed = judge_response.strip().startswith("GROUNDED")

    return {"key": "groundedness", "score": passed, "comment": judge_response}

## Evaluator 3: Format-Konsistenz

Prüft speziell beim Fact-Check-Tool, ob das erwartete BEWERTUNG/BEGRÜNDUNG/HINWEIS-Format eingehalten wurde.


In [10]:
def format_evaluator(inputs: dict, outputs: dict, reference_outputs: dict, run) -> dict:
    criteria = reference_outputs.get("criteria", "")

    # Nur relevant, wenn das Format im Kriterium erwähnt wird (also v.a. beim Fact-Check-Eintrag)
    if "format" not in criteria.lower():
        return {"key": "format_consistency", "score": None, "comment": "Kein Format-Kriterium für diese Frage"}

    answer = outputs.get("answer", "")
    has_all_parts = all(marker in answer for marker in ["BEWERTUNG", "BEGRÜNDUNG", "HINWEIS"])

    return {
        "key": "format_consistency",
        "score": has_all_parts,
        "comment": "Alle 3 Format-Marker gefunden" if has_all_parts else "Mindestens ein Marker fehlt",
    }

## Evaluation ausführen

Lässt den Agent gegen alle 6 Dataset-Einträge laufen und wendet alle 3 Evaluatoren an. Ergebnis landet automatisch auch in der LangSmith-Oberfläche.


In [11]:
from langsmith import evaluate

results = evaluate(
    target,
    data=dataset_name,
    evaluators=[
        tool_selection_evaluator,
        groundedness_evaluator,
        format_evaluator,
    ],
    experiment_prefix="health-fitness-eval",
)

print("✅ Evaluation abgeschlossen")

/Users/skander/Documents/IronHack/W8/health-fitness-qa-bot/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'health-fitness-eval-304a0e88' at:
https://smith.langchain.com/o/91ac1afe-2d87-486f-987c-ad935a1c1495/datasets/5c583561-6e74-4733-bffa-bca2a1519175/compare?selectedSessions=82c7b03c-f390-40bd-a865-4aafe5a04819




8it [00:45,  5.64s/it]

✅ Evaluation abgeschlossen


## Iteration


## Jetzt zu M7 — Fact-Check-Prompt schärfen


Warum das vermutlich manchmal nicht befolgt wird: LLMs neigen dazu, Format-Anweisungen in Prompts zu "vergessen" oder zu lockern, besonders wenn die Formulierung eher wie eine Empfehlung klingt ("Antworte in genau diesem Format") statt wie eine strikte Regel. Ein bekannter, effektiver Fix: die Anweisung noch expliziter und mit Wiederholung/Betonung formulieren.

Was sich geändert hat: Explizites "MUSS EXAKT", "ohne einleitende Sätze davor", und die zusätzliche Anweisung "Beginne IMMER mit 'BEWERTUNG:' als allererstes Wort" — das gibt dem LLM einen sehr konkreten, überprüfbaren Ankerpunkt, statt einer allgemeinen Formatvorgabe.

Folgende Prompt wird in tools.py erstzt (im fact_check_tool)


In [12]:
prompt = f"""Du bist ein wissenschaftlicher Fact-Checker im Bereich Ernährung/Fitness.

Folgende Aussage stammt aus einem YouTube-Video:
"{video_context}"

WICHTIG: Deine Antwort MUSS EXAKT diesem Format folgen, ohne Abweichung, ohne einleitende Sätze davor:

BEWERTUNG: [Weitgehend bestätigt / Teilweise bestätigt / Umstritten / Nicht ausreichend belegt]
BEGRÜNDUNG: [2-3 Sätze, warum]
HINWEIS: Diese Einschätzung basiert auf allgemeinem KI-Wissen, nicht auf einer geprüften externen Datenbank.

Beginne deine Antwort IMMER mit "BEWERTUNG:" als allererstes Wort. Füge KEINEN Text vor oder nach diesem Format hinzu."""

NameError: name 'video_context' is not defined

## Debug-Check: Ursache für groundedness=0.0 bei der Ballaststoffe-Frage

Schaut sich den tatsächlichen `context`-Wert (das, was an den Fact-Check-LLM ging) für den
fehlgeschlagenen Eval-Eintrag an, um zu bestätigen, dass `search_video()` themafremde
Chunks (Omega-3/6 statt Ballaststoffe) geliefert hat, bevor wir den Fix angehen.


In [ ]:
import pandas as pd, json

# Pfad ggf. anpassen, falls die CSV woanders liegt (z.B. Downloads-Ordner)
df = pd.read_csv("health-fitness-eval-9534da95.csv")
row = df[df['inputs'].str.contains("fiber")].iloc[0]
outputs = json.loads(row['outputs'])
print(outputs['context'][:500])

BEWERTUNG: Teilweise besttigt
BEGRÜNDUNG: Die Aussage, dass die meisten Menschen ausreichend Omega-6-Fettsäuren konsumieren, ist weitgehend korrekt, da diese in vielen pflanzlichen Ölen und verarbeiteten Lebensmitteln enthalten sind. Allerdings ist die Behauptung, dass die meisten Menschen nicht genug Omega-3-Fettsäuren zu sich nehmen, ebenfalls zutreffend, da viele westliche Ernährungsweisen arm an diesen Fettsäuren sind, die für die Gehirnfunktion wichtig sind. Die Empfehlung, fermentierte Le


## Threshold-Kalibrierung für search_video()

Misst Distance-Werte für einen bekannten guten Treffer (Kreatin, Video behandelt das)
und einen bekannten schlechten Treffer (Fiber, Video behandelt das nicht), um einen
sinnvollen max_distance-Wert für search_video() zu bestimmen -- analog zum KB-Threshold.


In [ ]:
from tools import embed_query, collection

for query in ["creatine cognition", "fiber intake"]:
    emb = embed_query(query)
    results = collection.query(query_embeddings=[emb], n_results=3, include=["documents", "distances"])
    print(f"--- {query} ---")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(round(dist, 3), doc[:60])
    print()

--- creatine cognition ---
0.77 in people that aren't getting creatine from animal sources. 
0.812 The first author is Roschel, R-O-S-C-H-E-L. We will provide 
0.866 But because, fortunately, at least, not yet, or not to my aw

--- fiber intake ---
1.183 throughout our entire life, and given that everybody has to 
1.254 I eat a fairly limited amount of meat. I don't restrict it, 
1.266 vegetables, including beans, cabbage, once again, spinach, p



## Check 1: Warum ist groundedness beim Fiber-Eintrag immer noch 0.0?

Schaut sich context + answer aus DIESEM (neuen) Eval-Lauf an, um zu sehen ob der Fix
tatsächlich gegriffen hat oder ob was anderes das Ergebnis verfälscht.


In [16]:
import json
import pandas as pd
df = pd.read_csv("health-fitness-eval-304a0e88.csv")

row = df[df['inputs'].str.contains("fiber")].iloc[0]
outputs = json.loads(row['outputs'])
print("ANSWER:", outputs['answer'][:400])
print()
print("CONTEXT:", outputs['context'][:400])


ANSWER: BEWERTUNG: Weitgehend bestätigt  
BEGRÜNDUNG: Die externe Evidenz belegt, dass ein erheblicher Teil der Bevölkerung die empfohlenen Ballaststoffzufuhren nicht erreicht, was zu gesundheitlichen Bedenken führt. Dies unterstützt die Aussage, dass das Thema der Ballaststoffaufnahme von großer Bedeutung ist, auch wenn das Video nicht spezifisch darauf eingeht.  
HINWEIS: Quelle: Live-Websuche (Tavily) 

CONTEXT: BEWERTUNG: Weitgehend bestätigt
BEGRÜNDUNG: Die externe Evidenz belegt, dass ein erheblicher Teil der Bevölkerung die empfohlenen Ballaststoffzufuhren nicht erreicht, was zu gesundheitlichen Bedenken führt. Dies unterstützt die Aussage, dass das Thema der Ballaststoffaufnahme von großer Bedeutung ist, auch wenn das Video nicht spezifisch darauf eingeht.
HINWEIS: Quelle: Live-Websuche (Tavily) -- n


## Check 2: Was ist beim "Who made this video"-Eintrag passiert?


In [17]:
row2 = df[df['inputs'].str.contains("Who made")].iloc[0]
print(row2['status'])
print(row2['error'])
print(row2['outputs'])

success
nan
{"answer":"Das Video mit dem Titel \"Nutrients For Brain Health & Performance | Huberman Lab Podcast #42\" wurde von Andrew Huberman erstellt und hat eine Länge von 101 Minuten.","context":"Titel: Nutrients For Brain Health & Performance | Huberman Lab Podcast #42\nKanal: Andrew Huberman\nHochgeladen am: 2021-10-18\nLänge: 101 Minuten\nThemen: Food and Brain, Cognition, Gut Signals, Metabolic Accessibility, Intermittent Fasting\nBeschreibung (Auszug): This episode I describe science-supported nutrients for brain and performance (cognition) and for nervous system health generally.\n\nI describe 10 tools for this purpose, including specific amounts and sources for Omega-3 fatty acids which make up the \"structural fat\" of neurons (nerve cells) and all...","tools_used":["get_video_metadata_tool"]}
